# 05 商品关联与类别预测

本 Notebook 对应阶段 3：问题二建模。每段代码前说明要解决的问题，每段代码后说明输出如何理解。

## 代码 1：读取阶段 1 处理后的数据

这段代码解决“使用什么销量口径和字段”的问题。问题二继续使用 `positive_sales` 表示正向顾客需求销量，并用 `product_id` 合并同一商品编号下的名称差异。

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
df = pd.read_csv(ROOT / 'data/processed/modeling_base_table.csv')
df['date'] = pd.to_datetime(df['date'])
target = 'positive_sales'
df[['date','store_id','store_name','product_id','product_name','category',target]].head()


输出理解：如果能看到日期、门店、商品、类别和 `positive_sales`，说明问题二所需字段齐全。若缺少 `category`，才需要根据商品名称或相关性重新分类；本题不需要。

## 代码 2：构造日期 × 商品销量矩阵

这段代码解决“商品之间如何放到同一张表比较”的问题。矩阵的每一列是一种商品的日销量序列。

In [ ]:
product_daily = df.groupby(['date','product_id'], as_index=False)[target].sum()
matrix = product_daily.pivot_table(index='date', columns='product_id', values=target, aggfunc='sum', fill_value=0)
matrix.head()


输出理解：每一行是一天，每一列是一种商品。某个单元格为 0 表示该商品当天没有正向销售。该矩阵用于后续计算相关系数。

## 代码 3：计算商品相关系数矩阵

这段代码解决“商品销量之间的联系如何量化”的问题。这里使用 Pearson 相关系数衡量两个商品日销量是否同步波动。

In [ ]:
corr = matrix.corr(method='pearson')
corr.round(3)


输出理解：相关系数接近 1 表示两个商品更同步，接近 0 表示线性关系弱，小于 0 表示反向波动线索。相关性不等于因果性，不能直接证明替代或带动关系。

## 代码 4：按类别聚合销量

这段代码解决“如何整合同类零食”的问题。因为附件已有 `category` 字段，所以直接按类别求和。

In [ ]:
category_daily = df.groupby(['date','category'], as_index=False)[target].sum()
category_stats = category_daily.groupby('category')[target].agg(['sum','mean','std']).sort_values('sum', ascending=False)
category_stats


输出理解：`sum` 表示类别历史累计销量，`mean` 表示类别平均日销量，`std` 表示日销量波动。类别聚合通常比单品更平滑，但会损失类别内部商品差异。

## 代码 5：读取阶段 3 已生成的预测比较结果

这段代码解决“聚合预测是否优于单品加总”的问题。完整滚动验证已写入项目输出表，这里读取结果进行论文解释。

In [ ]:
method_metrics = pd.read_csv(ROOT / 'tables/q2_category_method_metrics.csv')
method_metrics[['method_label','model_label','MAE','RMSE','WAPE_pct']]


输出理解：WAPE 越小，说明总绝对误差占真实销量的比例越低。该表用于比较“单品预测后加总”和“类别聚合后直接预测”的优劣。

## 代码 6：读取未来 7 天类别预测结果

这段代码解决“问题二最终要提交什么预测结果”的问题。预测日期为 2022-04-01 至 2022-04-07。

In [ ]:
forecast = pd.read_csv(ROOT / 'tables/q2_category_forecast_7day_total.csv')
forecast


输出理解：`predicted_7day_sales` 是该类别未来 7 天预测总销量。该表可直接用于问题二结果表，门店-类别版本见 `q2_store_category_forecast_7day_total.csv`。